# Week 13 — Multi-Agent RL and AlphaZero · Workbook

10 graded exercises (★ to ★★★). Reference solutions in `project/solutions.py`.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))
from common.plotting import use_rl_style, RL_PALETTE
from common.seed import seed_everything
use_rl_style(); seed_everything(0)
import numpy as np
import torch
from project import solutions as sol


## Exercise 1 — PUCT formula (★)

Implement the PUCT score: $-Q(c) + c_{\text{puct}} \cdot P(c) \cdot \sqrt{N(\text{parent})} / (1 + N(c))$.

> **Hint**: Q is negated because the child's value is from the opponent's POV.

In [ ]:
s = sol.exercise_1_puct(prior=0.5, parent_visits=100, child_visits=10, q=0.2, c_puct=1.5)
print('PUCT =', round(s, 3))
assert s > 0


## Exercise 2 — Terminal value (★)

Given a Tic-Tac-Toe board (canonical form: +1 = player to move, -1 = opponent),
return the terminal value from the current player's POV or `None` if the
game is ongoing.

> **Hint**: see `project.tictactoe.TicTacToe.terminal_value`.

In [ ]:
import numpy as np
state = np.zeros((3, 3), dtype=np.int8)
print('empty board:', sol.exercise_2_terminal_value(state))
state[0, :] = -1
print('row of O (just-moved player won):', sol.exercise_2_terminal_value(state))


## Exercise 3 — D4 symmetries (★)

Tic-Tac-Toe has 4 rotations × 2 reflections = 8 symmetries. Return the
number of augmented (state, pi) pairs.

> **Hint**: 4 rotations × 2 flips = 8.

In [ ]:
state = np.zeros((3, 3), dtype=np.int8)
pi = np.ones(9) / 9.0
n = sol.exercise_3_n_symmetries(state, pi)
print('# symmetries =', n)
assert n == 8


## Exercise 4 — MCTS visit distribution (★★)

Build a fresh `AZNetwork`, run MCTS for 30 simulations from the empty
board, return the visit distribution. It should be a valid probability
vector.

> **Hint**: the centre square typically gets the most visits for an untrained network because the prior is roughly uniform but the centre maximises subsequent line opportunities.

In [ ]:
pi = sol.exercise_4_mcts_visits(n_simulations=30, seed=0)
print('visit distribution:', np.round(pi, 3))
assert abs(pi.sum() - 1.0) < 1e-6


## Exercise 5 — AlphaZero winrate (★★)

Train AlphaZero for 5 short iterations; report winrate vs random opponent.
With 5 iterations and 20 simulations per move, winrate should be at least
0.5 (better-than-random).

> **Hint**: the full pipeline is in `project.train.train_alphazero`.

In [ ]:
wr = sol.exercise_5_alphazero_winrate(n_iters=5, seed=0)
print('AZ winrate:', round(wr, 2))
assert wr >= 0.5


## Exercise 6 — Generalised Advantage Estimation (★★)

Implement GAE on a simple sequence of rewards $r$ and value estimates $V$.
Verify that returns equal advantages + values.

> **Hint**: $A_t = \delta_t + \gamma \lambda A_{t+1}$, $\delta_t = r_t + \gamma V_{t+1} - V_t$.

In [ ]:
adv, ret = sol.exercise_6_gae([1, 0, 1, 0, 1], [0.5, 0.5, 0.5, 0.5, 0.5])
print('advantages:', np.round(adv, 3))
print('returns:', np.round(ret, 3))
assert np.allclose(ret - adv, 0.5)


## Exercise 7 — MAPPO on the coordination game (★★)

Train MAPPO on a 2-agent matrix coordination game (3 actions, 5 steps,
reward = +1 if actions match). Verify mean of last 30 episodes is at least
5 (out of max 10).

> **Hint**: in equilibrium both agents should always pick the same action; perfect coordination yields return = 2 × 5 = 10.

In [ ]:
ret_mean = sol.exercise_7_coord_game_return(n_episodes=200, seed=0)
print('mean return last 30 eps:', round(ret_mean, 2))
assert ret_mean >= 5.0


## Exercise 8 — MAPPO on SimpleSpread (★★)

Train MAPPO for a short run on the 3-agent SimpleSpread environment.
Compare first-10 and last-10 episode means. The cooperative-coverage
problem is hard; meaningful convergence requires ~10k+ episodes (published
benchmarks).

> **Hint**: published SimpleSpread results need ~10k episodes; this is a snapshot of early learning.

In [ ]:
first, last = sol.exercise_8_simple_spread(n_episodes=30, seed=0)
print(f'first 10 mean: {first:.1f}, last 10 mean: {last:.1f}')


## Exercise 9 — Self-play vs.\ fixed opponent (★★★, open-ended)

Argue (in a markdown cell) why self-play produces stronger policies than
training against a single fixed weak opponent. Use the cell below to run a
short A/B comparison.

> **Hint**: the moving-target problem; if the opponent never improves, your policy plateaus at ``barely beating the weak opponent''.

In [ ]:
d = sol.exercise_9_self_play_vs_fixed(seed=0)
print('self_play_winrate:', d['self_play_winrate'])
print('untrained_winrate:', d['untrained_winrate'])


**Discussion:** Self-play continuously raises the bar — every iteration the
opponent is at the same level as your current policy. Against a fixed weak
opponent, your policy converges to ``the simplest policy that beats this
opponent'', which can be arbitrarily far from a Nash strategy that
generalises.

## Exercise 10 — AlphaZero rating climb (★★★, open-ended)

Track winrate vs random across iterations. Plot the climb. Discuss: how
does the slope of the climb relate to the MCTS simulation budget?

> **Hint**: more simulations → bigger policy improvement per iteration → steeper climb. But also: more compute per iteration → fewer iterations in fixed time.

In [ ]:
wrs = sol.exercise_10_az_rating_climb(n_iters=6, seed=0)
print('winrates:', np.round(wrs, 2))
import matplotlib.pyplot as plt
plt.figure(figsize=(6, 3.5))
plt.plot(np.arange(1, len(wrs)+1), wrs, 'o-')
plt.axhline(0.5, color='gray', linestyle='--', label='random')
plt.xlabel('iteration'); plt.ylabel('winrate')
plt.title('AlphaZero rating climb on Tic-Tac-Toe')
plt.legend()
plt.show()


## Wrap-up

You have:
1. Implemented PUCT and MCTS from scratch.
2. Trained an AlphaZero network to beat random opponents on Tic-Tac-Toe.
3. Implemented MAPPO with centralised critic + decentralised execution.
4. Verified MAPPO converges on a 2-agent coordination game.
5. Studied the failure modes of MARL: non-stationarity, cycle dynamics,
   coordination failure.

Next week: offline RL (CQL, IQL, BCQ) --- moving from interactive learning
back to learning from logs.